# scEEMS Model Training

This notebook trains a single-cell Enhanced Modifier Score (scEEMS) model: a feature-weighted CatBoost classifier that prioritises non-coding regulatory variants in cell-type-specific contexts. It wraps the `gems_pipeline.py train` engine as a reproducible SoS workflow.

## Description

**Aim.** Train a cell-type-specific scEEMS model that bridges non-coding GWAS variants and their target genes by integrating Activity-by-Contact (ABC) enhancer-gene linkages with distance and evolutionary-constraint features. The approach aims to improve understanding of disease mechanisms in Alzheimer's disease.

**Method.** A [CatBoost](https://github.com/catboost/catboost) gradient-boosting classifier is trained with 10x upweighting of deep-learning (DL-VEP) features relative to other features. This configuration was selected as optimal from external validation and heritability analysis in the manuscript.

**Training data construction.** Fine-mapped single-cell eQTLs from six brain cell types in the ROSMAP cohort. Positive class (Y=1): variants with PIP > 0.05 in a credible set whose maximum PIP exceeds 0.1, or variants with PIP > 0.5 regardless of credible-set membership. Negative class (Y=0): for each positive variant, 10 negative variants are sampled from the same gene with PIP < 0.01, matched on variant type (SNP, insertion, deletion).

**Code repository.** [StatFunGen/xqtl-protocol — xqtl_modifier_score](https://github.com/StatFunGen/xqtl-protocol/tree/main/code/xqtl_modifier_score)

**Timing:** ~varies on typical compute infrastructure.

## Input

- `data_config.yaml` — data configuration: cohort, paths to fine-mapped eQTL tables, and the full feature set (4,839 features across the categories below). New cohorts/cell types are added by editing this YAML only.
- `model_config.yaml` — model configuration: CatBoost hyperparameters and the DL-VEP feature-weighting scheme.
- Gene-constraint table (e.g. `data/41588_2024_1820_MOESM4_ESM.xlsx`) providing GeneBayes constraint scores.

### Feature categories (4,839 total)

1. **Distance features** — physical proximity determines regulatory potential. e.g. `abs_distance_TSS_log` (log-transformed distance to transcription start site).
2. **Cell-type regulatory features** — functional-genomics assays for cell-specific regulation. e.g. `ABC_score_microglia` (Activity-by-Contact regulatory activity score).
3. **Population-genetics features** — minor-allele frequency can impact regulatory activity. e.g. `gnomad_MAF`.
4. **Conservation features** — evolutionary constraint across species.

## Output

- A trained feature-weighted CatBoost model (e.g. `results/model.cbm` / `.joblib`) for the requested cohort and chromosome.
- Performance metrics for the trained model. Reference single-model result (Model 5, feature-weighted): Average Precision (AP) ≈ 0.5050, AUC ≈ 0.8978.
- A ranked feature-importance table; top contributors include `abs_distance_TSS_log`, several `diff_32_ENCFF*` epigenomic tracks, `gnomad_MAF`, and `ABC_score_microglia`.

The trained model is consumed downstream by the prediction workflow — see [EMS Prediction](https://statfungen.github.io/xqtl-protocol/code/xqtl_modifier_score/ems_prediction.html) for variant scoring.

## Steps

**Step 1.** Train a feature-weighted CatBoost scEEMS model for one cohort / chromosome. The toy command below trains on the microglia cohort (`protocol_example`) for chromosome 2 using the bundled YAML configs.

In [ ]:
sos run pipeline/ems_training.ipynb train \
    --cohort protocol_example \
    --chromosome 2 \
    --data-config code/xqtl_modifier_score/data_config.yaml \
    --model-config code/xqtl_modifier_score/model_config.yaml \
    --cwd output/ems_training


In [ ]:
# Load gene constraint data
import pandas as pd
gene_data = pd.read_excel('data/41588_2024_1820_MOESM4_ESM.xlsx')
print(gene_data.head(3))

## Command interface

In [ ]:
sos run code/xqtl_modifier_score/ems_training.ipynb -h

In [ ]:
# Configuration file check
import os

required_files = [
    'gems_pipeline.py',  
    'data_config.yaml',
    'model_config.yaml'
]

print("Configuration File Check:")
print("=" * 50)
for file in required_files:
    status = "✅ Found" if os.path.exists(file) else "❌ Missing"
    print(f"  {file}: {status}")

## Workflow implementation

The `train` step wraps the `gems_pipeline.py train` engine. The pipeline automatically loads the training data, trains the feature-weighted CatBoost model, and writes the trained model and metrics. New datasets / cell types are added by editing the YAML configs only — no Python code changes are required (e.g. swap `protocol_example` for `Ast_mega_eQTL` with the matching `data_config` to train on astrocytes).

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
# Performance results 
print("Model Performance Results:")
print("=" * 50)

# Single model results (corrected from previous 4-model approach)
corrected_results = {
    'Feature-Weighted Model (Model 5)': {'AP': 0.5050, 'AUC': 0.8978}
}

for model, metrics in corrected_results.items():
    print(f"{model}:")
    print(f"  Average Precision: {metrics['AP']:.4f}")
    print(f"  AUC Score: {metrics['AUC']:.4f}")
    print()

print("Feature Importance (Top 10):")
print("-" * 40)
top_features = [
    ("abs_distance_TSS_log", 23.58),
    ("diff_32_ENCFF140MBA", 1.83),
    ("diff_32_ENCFF457MJJ", 1.12),
    ("gnomad_MAF", 0.88),
    ("diff_32_ENCFF455KGB", 0.84),
    ("diff_32_ENCFF579IKK", 0.80),
    ("ABC_score_microglia", 0.54),
    ("microglia_enhancer_promoter_union_atac", 0.58),
    ("diff_32_ENCFF098QMC", 0.66),
    ("diff_32_ENCFF757EYT", 0.66)
]

for i, (feature, importance) in enumerate(top_features, 1):
    print(f"{i:2d}. {feature:<35} {importance:>6.2f}")